---
title: "Private Estimation - cont — Self-study notebook"
subtitle: "Applied Data Privacy"
format:
  html:
    page-layout: full
    toc: false
    toc-depth: 2
    code-tools: true
    code-fold: true
    code-overflow: wrap
    include-in-header:
      text: |
        <style>
        .cell-output-display img, .cell-output-display .plotly-graph-div { max-width: 100%; height: auto; }
        </style>
execute:
  enabled: true
  warning: false
  message: false
jupyter: python3
---

This is the **self-study** companion to the lecture deck: full narrative + all code, meant to be
read and run. For the slide version (code hidden), open the [presentation deck](../../lecture-presentations/private-subgroup-comparisons/presentation.html).


In [ ]:
#| echo: false
#| output: false
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy[notebook] @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy

%matplotlib inline


In [ ]:
#| echo: false
#| output: false
import pandas as pd

from libdpy.assignment_specific.private_subgroup_comparisons.lecture_figures import (
    make_above_threshold_support_search_figure,
    make_global_sensitivity_baseline_figure,
    make_local_sensitivity_vs_support_figure,
    make_noisy_count_sum_release_figure,
    make_oracle_ls_neighbor_bars_figure,
    make_oracle_ls_privacy_failure_figure,
    make_oracle_ls_utility_figure,
    make_oracle_vs_count_sum_audit_figure,
    make_sparse_subgroup_warning_figure,
    make_subgroup_mechanism_leaderboard_figure,
    make_subgroup_proposals_diagnoses_repairs_table,
    make_support_comparison_figure,
    make_support_repair_leaderboard_figure,
    make_subgroup_sampling_distribution_figure,
)

SEED = 42


# Private Estimation - cont

**Thesis:** After a private step has controlled the salary range, subgroup analysis is about **denominators and private control flow**, not extreme outliers.

**Public contract:** Lecture 5 released a clipping interval \([L,U]\). From here on it is public. We work on normalized clipped salaries
\[
x_i = \frac{\mathrm{clip}(y_i,L,U)-L}{U-L}\in[0,1],
\]
and post-process dollar-scale estimates by \((U-L)\widehat{\Delta}_x\).

**Target statistic:** difference of two normalized group means on clipped salaries. Dollar-scale estimates are post-processing: multiply by \((U-L)\).


## Proposals, diagnoses, and repairs

Same rhythm as Lecture 5: tempting proposal → witness → diagnosis → valid repair.

In [ ]:
proposal_rows = make_subgroup_proposals_diagnoses_repairs_table()
pd.DataFrame(proposal_rows)


## Sampling variability before privacy

Ordinary statistical variability for the subgroup difference before adding privacy noise.

In [ ]:
#| echo: false
fig = make_subgroup_sampling_distribution_figure(seed=SEED)
fig


## Global sensitivity is pessimistic

If tiny groups are allowed in the domain, \(GS_\Delta \le 2\) on the normalized scale. A global-sensitivity Laplace mechanism adds noise at scale roughly \(2/\varepsilon\), which is too large for useful subgroup analysis on balanced samples.

**Diagnosis:** the worst case is engineered sparse support.

In [ ]:
#| echo: false
fig = make_global_sensitivity_baseline_figure(seed=SEED)
fig


## Local sensitivity explains typical utility

Once values live in \([0,1]\), accuracy is controlled by denominators:
\[
LS_\Delta(D) \approx \frac{1}{n_A}+\frac{1}{n_B}.
\]
Subgroup comparison is a denominator problem.

In [ ]:
#| echo: false
fig = make_local_sensitivity_vs_support_figure(seed=SEED)
fig


## Temptation: scale noise by local sensitivity

Scale noise by \(LS_\Delta(D)\). Looks excellent on balanced datasets — **not DP**. Local sensitivity is an excellent diagnostic, but a bad noise scale.

In [ ]:
#| echo: false
fig = make_oracle_ls_utility_figure(seed=SEED)
fig


### Engineered sparse support

Same 990/10 split as Lecture 5: denominator-fragile even before privacy.

In [ ]:
#| echo: false
fig = make_sparse_subgroup_warning_figure(seed=SEED)
fig


## Why the tempting mechanism is not private

Neighboring datasets can change the **noise scale** when support drops. The output distribution leaks subgroup support.

In [ ]:
#| echo: false
fig = make_oracle_ls_neighbor_bars_figure(seed=SEED)
fig


### Audit the released estimate

The privacy failure is visible in the output distribution, not only in the internal scale.

In [ ]:
#| echo: false
fig = make_oracle_ls_privacy_failure_figure(seed=SEED)
fig


### Same witness, valid repair

Noisy count/sum spends budget on denominators instead of adapting the noise scale for free.

In [ ]:
#| echo: false
fig = make_oracle_vs_count_sum_audit_figure(seed=SEED)
fig


## Valid baseline: release counts and sums

Four Laplace queries + public denominator floor \(\tau\). Counts are not free, but sometimes they are the right output.

In [ ]:
#| echo: false
fig = make_noisy_count_sum_release_figure(seed=SEED)
fig


## PTR and smooth sensitivity (conceptual)

**PTR** tests distance from datasets with \(\min(n_A,n_B) < m\), then releases or abstains. **Smooth sensitivity** grows noise smoothly near sparse regions — shown here as a conceptual comparison, not a formal pure-DP guarantee.

## Common support comparison

Vary true minimum support \(\min(n_A, n_B)\) — not only the PTR threshold \(m\).

In [ ]:
#| echo: false
fig = make_support_comparison_figure(seed=SEED)
fig


### Sparse vs dense support repairs

Conditional PTR error is only shown where releases are common enough to estimate honestly.

In [ ]:
#| echo: false
fig = make_support_repair_leaderboard_figure(seed=SEED)
fig


### Mechanism leaderboard on Fulton

Same Lecture 5 signed-error workflow, now comparing subgroup mechanisms.

In [ ]:
#| echo: false
fig = make_subgroup_mechanism_leaderboard_figure(seed=SEED)
fig


## Public data gives a menu

Public coarsenings and subgroup definitions are free to propose. Private validation of support costs privacy budget.

In [ ]:
#| echo: false
fig = make_above_threshold_support_search_figure(seed=SEED)
fig


## Summary

After clipping, subgroup analysis is about **denominators and private control flow**. Public coarsenings propose questions; private validation of support costs budget.